# Fine-Tuning vs. In-Context Learning for Political Bias Classification

**Authors:** Eitan Derdiger, Yedidya Levine, Shira Yogev  
**Course:** Advanced Models of Language Understanding - Spring 2026

This notebook contains the complete experimental pipeline used in the project. It combines the two raw dataset files, cleans and splits the data, builds classical baselines, fine-tunes Qwen2.5-0.5B-Instruct with LoRA, evaluates Qwen and Gemma with in-context learning (ICL), aggregates predictions, computes metrics, and produces the plots used in the analysis.

The expensive experiment switches are disabled by default. The final prediction files, metrics, and plots used in the report are already included in the repository.

## Repository layout

The notebook is designed for the following repository structure:

```text
project-root/
├── code/
│   ├── Political_Bias_FT_vs_ICL.ipynb
│   └── requirements.txt
├── data/
│   ├── Political_Bias.csv
│   ├── Political_Bias_Update.csv
│   ├── political_bias_combined.csv
│   └── political_bias_cleaned.csv
├── results/
├── plots/
├── Final_Project_Report.pdf
└── README.md
```

Run the notebook from either the repository root or the `code/` directory. Model downloads and LoRA adapters are stored under `.artifacts/`, which is intentionally excluded from Git.

## 1. Setup and configuration

Install the dependencies from `code/requirements.txt` before running the notebook. GPU-backed execution is strongly recommended for the Fine-Tuning and ICL sections.

In [ ]:
import gc
import os
import platform
import random
import re
import statistics
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from huggingface_hub import snapshot_download
from peft import LoraConfig, PeftConfig, PeftModel, get_peft_model
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer, EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

In [ ]:
# Keep Hugging Face downloads predictable and avoid tokenizer multiprocessing warnings.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def find_project_root() -> Path:
    """Locate the repository root from the current working directory."""
    explicit_root = os.getenv("POLITICAL_BIAS_PROJECT_DIR")
    if explicit_root:
        return Path(explicit_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]

    for candidate in candidates:
        if (candidate / "data").is_dir() and (candidate / "code").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the repository root. Run the notebook from the "
        "repository root or the code/ directory, or set POLITICAL_BIAS_PROJECT_DIR."
    )


PROJECT_DIR = find_project_root()
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
PLOTS_DIR = PROJECT_DIR / "plots"
ARTIFACTS_DIR = PROJECT_DIR / ".artifacts"
MODELS_DIR = ARTIFACTS_DIR / "base_models"
ADAPTERS_DIR = ARTIFACTS_DIR / "lora_adapters"

for directory in [DATA_DIR, RESULTS_DIR, PLOTS_DIR, MODELS_DIR, ADAPTERS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = DATA_DIR / "Political_Bias.csv"
RAW_UPDATE_PATH = DATA_DIR / "Political_Bias_Update.csv"
COMBINED_DATA_PATH = DATA_DIR / "political_bias_combined.csv"
CLEAN_DATA_PATH = DATA_DIR / "political_bias_cleaned.csv"

MODEL_PATHS = {
    "Qwen/Qwen2.5-0.5B-Instruct": MODELS_DIR / "Qwen2.5-0.5B-Instruct",
    "google/gemma-2-2b-it": MODELS_DIR / "gemma-2-2b-it",
}

print(f"Project root: {PROJECT_DIR}")

In [ ]:
# Expensive stages are disabled by default for a safe GitHub-ready notebook.
# Enable the stages you want before running the full pipeline.
RUN_FINE_TUNING = False
RUN_ICL = False
RUN_EVALUATION = False
RUN_EXPERIMENT_2 = False

FINE_TUNING_MODEL_NAMES = ["Qwen/Qwen2.5-0.5B-Instruct"]
ICL_MODEL_NAMES = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "google/gemma-2-2b-it",
]

MAX_LENGTH_FT = 2048
MAX_LENGTH_ICL = 4096
SEED = 42
TRAIN_SIZES = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
SHOTS = [0, 1, 3, 5, 10]
LABEL_ORDER = ["left", "lean left", "center", "lean right", "right"]

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

hardware_info = {
    "Python Version": platform.python_version(),
    "PyTorch Version": torch.__version__,
    "CUDA Available": torch.cuda.is_available(),
    "CUDA Version": torch.version.cuda,
    "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only",
    "Fine-Tuning Maximum Token Length": MAX_LENGTH_FT,
    "ICL Maximum Token Length": MAX_LENGTH_ICL,
    "Seed": SEED,
}

hardware_df = pd.DataFrame(hardware_info.items(), columns=["Setting", "Value"])
hardware_df.to_csv(RESULTS_DIR / "environment_info.csv", index=False)

print(hardware_df.to_string(index=False))

In [ ]:
def download_model_once(model_name: str) -> Path:
    """Download a model once and reuse the local copy on later runs."""
    model_path = MODEL_PATHS[model_name]
    config_file = model_path / "config.json"

    if config_file.exists():
        print(f"Using cached model: {model_path}")
        return model_path

    print(f"Downloading {model_name}...")
    snapshot_download(
        repo_id=model_name,
        local_dir=str(model_path),
        max_workers=1,
    )
    print(f"Model saved to: {model_path}")
    return model_path

## 2. Dataset preparation

### 2.1 Merge the original dataset files

This cell replaces the separate merge script from the original project. It keeps the exact same merge rule: concatenate both files and keep the last occurrence of every duplicated article link.

In [ ]:
raw_df = pd.read_csv(RAW_DATA_PATH)
update_df = pd.read_csv(RAW_UPDATE_PATH)

combined_df = (
    pd.concat([raw_df, update_df], ignore_index=True)
    .drop_duplicates(subset=["Link"], keep="last")
    .reset_index(drop=True)
)

combined_df.to_csv(COMBINED_DATA_PATH, index=False)

print(f"Original rows: {len(raw_df):,}")
print(f"Update rows:   {len(update_df):,}")
print(f"Combined rows: {len(combined_df):,}")
print(f"Saved: {COMBINED_DATA_PATH}")

### 2.2 Clean the combined dataset

The cleaning steps are unchanged from the final experiment notebook: remove missing/empty values, failed article downloads, and duplicate title-text pairs.

In [ ]:
dataset_df = combined_df.copy()
rows_before_cleaning = len(dataset_df)

# Remove rows that cannot be used as supervised examples.
dataset_df = dataset_df.dropna(subset=["Title", "Text", "Bias"]).copy()

for column in ["Title", "Text", "Bias"]:
    dataset_df[column] = dataset_df[column].astype(str).str.strip()

dataset_df = dataset_df[
    (dataset_df["Title"] != "")
    & (dataset_df["Text"] != "")
    & (dataset_df["Bias"] != "")
]

dataset_df = dataset_df[
    ~dataset_df["Text"].str.contains(
        "Error fetching article",
        case=False,
        na=False,
    )
]

dataset_df = (
    dataset_df
    .drop_duplicates(subset=["Title", "Text"], keep="first")
    .reset_index(drop=True)
)

rows_after_cleaning = len(dataset_df)
dataset_df.to_csv(CLEAN_DATA_PATH, index=False)

print(f"Rows before cleaning: {rows_before_cleaning:,}")
print(f"Rows after cleaning:  {rows_after_cleaning:,}")
print(f"Rows removed:         {rows_before_cleaning - rows_after_cleaning:,}")
print("\nClass distribution:")
print(dataset_df["Bias"].value_counts())
print(f"\nSaved: {CLEAN_DATA_PATH}")

X = "Title: " + dataset_df["Title"] + "\n\nArticle:\n" + dataset_df["Text"]
y = dataset_df["Bias"]

### 2.3 Train, validation, and test split

A stratified 70/15/15 split is used with the same fixed seed as the original experiments.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

### 2.4 Nested Fine-Tuning subsets

The smaller Fine-Tuning subsets are nested inside the larger subsets so that increasing the training percentage adds data instead of replacing the previous sample.

In [ ]:
train_sizes = TRAIN_SIZES
train_subsets = {}

nested_sizes = sorted(train_sizes, reverse=True)
parent_X, parent_y = X_train, y_train

for idx, size in enumerate(nested_sizes):
    if size == 1.0:
        subset_X, subset_y = parent_X, parent_y
    else:
        parent_size = nested_sizes[idx - 1]
        sample_fraction = size / parent_size
        subset_X, _, subset_y, _ = train_test_split(
            parent_X, parent_y,
            train_size=sample_fraction,
            random_state=SEED,
            stratify=parent_y,
        )

    train_subsets[size] = (subset_X, subset_y)
    parent_X, parent_y = subset_X, subset_y
    print(f"Train size {int(size * 100)}%: X={subset_X.shape}, y={subset_y.shape}")

#### Verify that the subsets are nested

In [ ]:
ascending_sizes = sorted(train_sizes)

for small_size, large_size in zip(ascending_sizes[:-1], ascending_sizes[1:]):
    small_X, _ = train_subsets[small_size]
    large_X, _ = train_subsets[large_size]

    is_nested = set(small_X.index).issubset(set(large_X.index))

    print(f"{int(small_size * 100)}% inside {int(large_size * 100)}%: {is_nested}")

## 3. Classical baselines

We use a majority-class baseline and TF-IDF with logistic regression. These provide simple reference points for the language-model experiments.

In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)
dummy_preds = dummy_clf.predict(X_test)

dummy_acc = accuracy_score(y_test, dummy_preds)
dummy_f1 = f1_score(y_test, dummy_preds, average="macro")

print(f"Majority Class Baseline - Accuracy: {dummy_acc:.4f}, Macro-F1: {dummy_f1:.4f}")

vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

lr_clf = LogisticRegression(max_iter=1000)
lr_clf.fit(X_train_tfidf, y_train)
lr_preds = lr_clf.predict(X_test_tfidf)

lr_acc = accuracy_score(y_test, lr_preds)
lr_f1 = f1_score(y_test, lr_preds, average="macro")

print(f"TF-IDF + Logistic Regression - Accuracy: {lr_acc:.4f}, Macro-F1: {lr_f1:.4f}")

del vectorizer, lr_clf, X_train_tfidf, X_test_tfidf, dummy_clf
gc.collect()

## 4. LoRA Fine-Tuning

Qwen2.5-0.5B-Instruct is fine-tuned with LoRA on six nested training-set sizes. The model architecture, LoRA configuration, training hyperparameters, and evaluation prompt are kept the same as in the final project experiments.

In [ ]:
def format_example(article, tokenizer, bias=None, max_length=MAX_LENGTH_FT):
    prefix = """Given an article and its title, classify the political bias of the article.

Possible labels:
left, lean left, center, lean right, right

"""
    suffix = "\n\nBias:"
    if bias is not None:
        suffix += f" {bias}"

    prefix_length = len(tokenizer.encode(prefix, add_special_tokens=False))
    suffix_length = len(tokenizer.encode(suffix, add_special_tokens=False))
    special_tokens = tokenizer.num_special_tokens_to_add(pair=False)

    article_budget = max_length - prefix_length - suffix_length - special_tokens
    if article_budget <= 0:
        raise ValueError("MAX_LENGTH_FT is too small for the prompt.")

    article_tokens = tokenizer.encode(
        str(article),
        add_special_tokens=False,
        truncation=True,
        max_length=article_budget,
    )

    shortened_article = tokenizer.decode(
        article_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    prompt = prefix + shortened_article + suffix

    while (
        len(tokenizer.encode(prompt, add_special_tokens=True, truncation=False)) > max_length
        and article_tokens
    ):
        article_tokens = article_tokens[:-1]
        shortened_article = tokenizer.decode(
            article_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        prompt = prefix + shortened_article + suffix

    final_length = len(tokenizer.encode(prompt, add_special_tokens=True, truncation=False))

    if final_length > max_length:
        raise ValueError(f"Formatted example has {final_length} tokens.")

    if bias is not None and not prompt.endswith(str(bias)):
        raise ValueError("The training label was lost.")

    if bias is None and not prompt.rstrip().endswith("Bias:"):
        raise ValueError("The Bias field was lost.")

    return prompt

In [ ]:
def fine_tune(
    model,
    tokenizer,
    dataset,
    validation_dataset=None,
    epochs=10,
    learning_rate=2e-4,
    output_dir="./fine_tuned_model",
):
    if not dataset:
        raise ValueError("Training dataset is empty")

    train_dataset = Dataset.from_dict({"text": [text.strip() for text in dataset]})
    eval_dataset = (
        Dataset.from_dict({"text": [text.strip() for text in validation_dataset]})
        if validation_dataset is not None
        else None
    )

    has_validation = eval_dataset is not None

    sft_config = SFTConfig(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=4,
        learning_rate=learning_rate,
        num_train_epochs=epochs,
        logging_strategy="epoch",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        packing=False,
        max_length=MAX_LENGTH_FT,
        dataset_text_field="text",
        eval_strategy="epoch" if has_validation else "no",
        save_strategy="epoch" if has_validation else "no",
        load_best_model_at_end=has_validation,
        metric_for_best_model="eval_loss" if has_validation else None,
        greater_is_better=False if has_validation else None,
    )

    callbacks = (
        [EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0)]
        if has_validation
        else []
    )

    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        processing_class=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        callbacks=callbacks,
    )

    trainer.train()
    trainer.save_model()

    return trainer.model, trainer

In [ ]:
def load_model_lora(
    model_path,
    tokenizer,
    lora_rank=8,
    lora_alpha=32,
    lora_dropout=0.05,
    gradient_checkpointing=True,
):
    """Load the base model and attach the LoRA adapters used in the project."""
    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        dtype = torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        str(model_path),
        local_files_only=True,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    )
    model.config.pad_token_id = tokenizer.pad_token_id

    if gradient_checkpointing:
        model.gradient_checkpointing_enable()
        model.config.use_cache = False
    else:
        model.config.use_cache = True

    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

In [ ]:
if RUN_FINE_TUNING:
    training_times = {}
    train_sizes = TRAIN_SIZES

    for model_name in FINE_TUNING_MODEL_NAMES:
        model_path = download_model_once(model_name)

        tokenizer = AutoTokenizer.from_pretrained(str(model_path), use_fast=True, local_files_only=True)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        tokenizer.padding_side = "right"

        formatted_val = [
            format_example(article, tokenizer, bias)
            for article, bias in zip(X_val, y_val)
            ]

        for train_size in train_sizes:
            print(f"\nTraining {model_name} with {train_size * 100:.0f}% of the training data")

            gc.collect()

            model = load_model_lora(model_path, tokenizer)

            train_set, labels = train_subsets[train_size]

            formatted_train = [
                format_example(article, tokenizer, bias)
                for article, bias in zip(train_set, labels)
            ]

            output_dir = (
                ADAPTERS_DIR / f"{model_name.replace('/', '_')}_"
                f"{int(train_size * 100)}_percent"
            )

            start_time = time.perf_counter()
            trained_model, trainer = fine_tune(
                model,
                tokenizer,
                formatted_train,
                validation_dataset=formatted_val,
                epochs=10,
                learning_rate=2e-4,
                output_dir=str(output_dir),
            )
            elapsed_time = time.perf_counter() - start_time

            experiment_name = f"{model_name}_{int(train_size * 100)}_percent"
            training_times[experiment_name] = elapsed_time

            training_times_df = pd.DataFrame([
                {
                    "Experiment": name,
                    "Training Time Seconds": seconds,
                    "Training Time Minutes": seconds / 60,
                }
                for name, seconds in training_times.items()
            ])
            training_times_df.to_csv(RESULTS_DIR / "training_times.csv", index=False)

            print(f"Training time: {elapsed_time / 60:.2f} minutes")

            del trained_model, trainer, model, training_times_df, formatted_train
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()

        del tokenizer, formatted_val

## 5. In-Context Learning

The ICL experiment uses Qwen2.5-0.5B-Instruct and Gemma 2 2B. Three independent 10-example demonstration sets are created, with two examples from each class. The 1-, 3-, and 5-shot settings use prefixes of those sets, matching the original experiment.

In [ ]:
def create_icl_sets(X_train, y_train, num_sets=3, examples_per_label=2, random_state=SEED):
    train_df = pd.DataFrame({
        "article": X_train.reset_index(drop=True),
        "bias": y_train.reset_index(drop=True),
    })

    labels = sorted(train_df["bias"].unique())
    required_per_label = num_sets * examples_per_label
    icl_sets = [[] for _ in range(num_sets)]

    for label in labels:
        label_examples = train_df[train_df["bias"] == label].sample(
            n=required_per_label,
            random_state=random_state,
            replace=False,
        ).reset_index(drop=True)

        for set_index in range(num_sets):
            start = set_index * examples_per_label
            selected = label_examples.iloc[start:start + examples_per_label]

            for _, row in selected.iterrows():
                icl_sets[set_index].append((row["article"], row["bias"]))

    for set_index, examples in enumerate(icl_sets):
        examples_df = pd.DataFrame(examples, columns=["article", "bias"])
        examples_df = examples_df.sample(frac=1, random_state=random_state + set_index)
        icl_sets[set_index] = list(examples_df.itertuples(index=False, name=None))

    return icl_sets

In [ ]:
icl_sets = create_icl_sets(
    X_train=X_train,
    y_train=y_train,
    num_sets=3,
    examples_per_label=2,
    random_state=SEED,
)

for index, example_set in enumerate(icl_sets, start=1):
    labels = [bias for _, bias in example_set]
    print(f"Set {index}: {len(example_set)} examples")
    print(pd.Series(labels).value_counts())

### 5.1 ICL prompt construction

In [ ]:
def shorten_text(text, tokenizer, max_tokens):
    tokens = tokenizer.encode(
        str(text),
        add_special_tokens=False,
        truncation=True,
        max_length=max_tokens,
    )

    return tokenizer.decode(
        tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )


def format_icl_prompt(examples, article, tokenizer, max_length=MAX_LENGTH_ICL):
    header = """You are a political bias classification assistant.

Your task is to classify the political bias of a news article.

Choose exactly one of the following labels:
- left
- lean left
- center
- lean right
- right

Below are some labeled examples.

"""
    target_prefix = """Now classify the following article:

"""
    target_suffix = """

Bias:"""

    fixed_prompt = header

    for i, (_, example_bias) in enumerate(examples, start=1):
        fixed_prompt += f"Example {i}:\n\n\n\nBias: {example_bias}\n\n"

    fixed_prompt += target_prefix + target_suffix

    fixed_tokens = len(
        tokenizer.encode(fixed_prompt, add_special_tokens=True, truncation=False)
    )
    available_tokens = max_length - fixed_tokens - 16

    if available_tokens <= 0:
        raise ValueError("Not enough space for the ICL prompt.")

    if not examples:
        target_budget = available_tokens
        example_budget = 0
    else:
        target_budget = min(max(200, available_tokens // 2), available_tokens)
        example_budget = max(1, (available_tokens - target_budget) // len(examples))

    prompt = header

    for i, (example_article, example_bias) in enumerate(examples, start=1):
        shortened_example = shorten_text(example_article, tokenizer, example_budget)
        prompt += f"""Example {i}:

{shortened_example}

Bias: {example_bias}

"""

    shortened_target = shorten_text(article, tokenizer, target_budget)
    prompt += target_prefix + shortened_target + target_suffix

    final_length = len(
        tokenizer.encode(prompt, add_special_tokens=True, truncation=False)
    )

    if final_length > max_length:
        raise ValueError(
            f"ICL prompt has {final_length} tokens, above MAX_LENGTH={max_length}."
        )

    if not prompt.rstrip().endswith("Bias:"):
        raise ValueError("The target Bias field was lost.")

    return prompt

In [ ]:
shots = SHOTS

### 5.2 Prediction parser

The parser below is intentionally kept identical to the parser used for the reported experiments so that the repository reproduces the same evaluation pipeline. It searches the complete generated text for label phrases in a fixed order.

In [ ]:
def parse_prediction(raw_text):
    text = str(raw_text).lower().strip()

    if "lean left" in text:
        return "lean left"
    elif "lean right" in text:
        return "lean right"
    elif "left" in text:
        return "left"
    elif "right" in text:
        return "right"
    elif "center" in text:
        return "center"

    return "invalid"


test_outputs = [
    "left",
    "  Bias: lean right\n",
    "I think it is center.",
    "completely unrelated text",
]

print("Testing parser:")
for out in test_outputs:
    print(f"'{out}' -> '{parse_prediction(out)}'")

## 6. Model inference

### 6.1 Evaluate Fine-Tuned adapters

This stage loads every LoRA adapter produced by the Fine-Tuning section and saves one prediction CSV per adapter.

In [ ]:
if RUN_EVALUATION:
    results_dir = ADAPTERS_DIR
    ft_predictions = {}
    ft_inference_times = {}

    base_model_name = FINE_TUNING_MODEL_NAMES[0]
    base_model_path = download_model_once(base_model_name)

    tokenizer = AutoTokenizer.from_pretrained(str(base_model_path), local_files_only=True)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    for adapter_dir in sorted(results_dir.iterdir()):
        if not adapter_dir.is_dir():
            continue

        if not (adapter_dir / "adapter_config.json").exists():
            print(f"Skipping {adapter_dir}: not a LoRA adapter")
            continue

        print(f"\nEvaluating adapter: {adapter_dir.name}")

        peft_config = PeftConfig.from_pretrained(str(adapter_dir))


        base_model = AutoModelForCausalLM.from_pretrained(
            str(base_model_path),
            local_files_only=True,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
            low_cpu_mem_usage=True,

        )

        model = PeftModel.from_pretrained(base_model, str(adapter_dir))
        model.requires_grad_(False)
        model.eval()

        predictions = []
        start_time = time.perf_counter()

        for article in X_test:
            prompt = format_example(article, tokenizer)
            inputs = tokenizer(prompt, return_tensors="pt", truncation=False).to(model.device)

            with torch.inference_mode():
                outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)

            generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
            prediction_text = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True,
            ).strip()

            predictions.append(parse_prediction(prediction_text))

        elapsed_time = time.perf_counter() - start_time

        if len(predictions) != len(y_test):
            raise ValueError(
                f"Expected {len(y_test)} predictions, but received {len(predictions)}."
            )

        ft_inference_times[adapter_dir.name] = elapsed_time
        ft_predictions[adapter_dir.name] = predictions

        print(f"Inference time: {elapsed_time:.2f} seconds")

        ft_output_df = pd.DataFrame({
            "True Label": y_test.reset_index(drop=True),
            "Predicted Label": predictions,
        })
        ft_output_df.to_csv(
            RESULTS_DIR / f"predictions_{adapter_dir.name}.csv",
            index=False,
        )

        ft_times_df = pd.DataFrame([
            {
                "Experiment": name,
                "Inference Time Seconds": seconds,
                "Seconds Per Article": seconds / len(y_test),
            }
            for name, seconds in ft_inference_times.items()
        ])
        ft_times_df.to_csv(RESULTS_DIR / "ft_inference_times.csv", index=False)

        del model, base_model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    del tokenizer
    gc.collect()

### 6.2 Run ICL inference

In [ ]:
if RUN_ICL:
    all_results = {}
    icl_inference_times = {}

    for model_name in ICL_MODEL_NAMES:
        print(f"\nRunning ICL for model: {model_name}")
        model_path = download_model_once(model_name)

        results = defaultdict(lambda: [[], [], []])
        tokenizer = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        tokenizer.padding_side = "left"

        model = AutoModelForCausalLM.from_pretrained(
            str(model_path),
            local_files_only=True,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
        model.eval()

        batch_size = 16 if "0.5B" in model_name else 4

        for shot in shots:
            set_indices = [0] if shot == 0 else range(3)

            for set_index in set_indices:
                start_time = time.perf_counter()
                examples = icl_sets[set_index][:shot]

                prompts = [
                    format_icl_prompt(examples, article, tokenizer)
                    for article in X_test
                ]

                predictions = []

                for batch_start in range(0, len(prompts), batch_size):
                    batch_prompts = prompts[batch_start:batch_start + batch_size]

                    inputs = tokenizer(
                        batch_prompts,
                        return_tensors="pt",
                        truncation=False,
                        padding=True,
                    ).to(model.device)

                    if inputs["input_ids"].shape[1] > MAX_LENGTH_ICL:
                        raise ValueError("An ICL batch exceeds MAX_LENGTH.")

                    with torch.inference_mode():
                        outputs = model.generate(
                            **inputs,
                            max_new_tokens=10,
                            do_sample=False,
                        )

                    input_length = inputs["input_ids"].shape[1]

                    for output in outputs:
                        generated_tokens = output[input_length:]
                        prediction = tokenizer.decode(
                            generated_tokens,
                            skip_special_tokens=True,
                        ).strip()
                        predictions.append(prediction)

                results[shot][set_index] = predictions
                elapsed_time = time.perf_counter() - start_time

                time_key = f"{model_name}_{shot}_shot_set_{set_index + 1}"
                icl_inference_times[time_key] = elapsed_time

                print(f"Inference time: {elapsed_time:.2f} seconds")

                if shot == 0:
                    print(f"Finished ICL zero-shot for {model_name}")
                else:
                    print(f"Finished ICL {shot}-shot (Set {set_index + 1}) for {model_name}")

                safe_model_name = model_name.replace("/", "_")

                raw_icl_df = pd.DataFrame({
                    "True Label": y_test.reset_index(drop=True),
                    "Raw Prediction": predictions,
                    "Parsed Prediction": [
                        parse_prediction(prediction) for prediction in predictions
                    ],
                })

                output_file = (
                    f"icl_predictions_{safe_model_name}_{shot}_shot_"
                    f"set_{set_index + 1}.csv"
                )
                raw_icl_df.to_csv(RESULTS_DIR / output_file, index=False)

        all_results[model_name] = dict(results)

        icl_times_df = pd.DataFrame([
            {
                "Experiment": name,
                "Inference Time Seconds": seconds,
                "Seconds Per Article": seconds / len(y_test),
            }
            for name, seconds in icl_inference_times.items()
        ])
        icl_times_df.to_csv(RESULTS_DIR / "icl_inference_times.csv", index=False)

        del model, tokenizer
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

## 7. Aggregate ICL predictions

For each nonzero-shot setting, the three prompt-set predictions are combined using the same `statistics.mode` rule as the original project. The original aggregation behavior is retained for reproducibility.

In [ ]:
if RUN_EVALUATION:
    icl_predictions = {}

    if "all_results" in globals():
        for model_name, shots_data in all_results.items():
            icl_predictions[model_name] = {}

            for shot, sets_data in shots_data.items():
                clean_sets = {}

                for set_index, predictions in enumerate(sets_data):
                    if predictions:
                        clean_sets[set_index] = [
                            parse_prediction(prediction) for prediction in predictions
                        ]

                if not clean_sets:
                    print(f"No predictions found for {model_name}, {shot}-shot")
                    continue

                num_articles = len(next(iter(clean_sets.values())))
                final_predictions = []

                for article_index in range(num_articles):
                    votes = [
                        predictions[article_index]
                        for predictions in clean_sets.values()
                    ]

                    try:
                        majority_vote = statistics.mode(votes)
                    except statistics.StatisticsError:
                        majority_vote = votes[0]

                    final_predictions.append(majority_vote)

                icl_predictions[model_name][shot] = final_predictions

                if len(final_predictions) != len(y_test):
                    raise ValueError(
                        f"{model_name}, {shot}-shot: expected {len(y_test)} predictions, "
                        f"but received {len(final_predictions)}."
                    )

                safe_model_name = model_name.replace("/", "_")

                aggregated_output_df = pd.DataFrame({
                    "True Label": y_test.reset_index(drop=True),
                    "Predicted Label": final_predictions,
                })

                output_file = f"icl_aggregated_predictions_{safe_model_name}_{shot}_shot.csv"
                aggregated_output_df.to_csv(RESULTS_DIR / output_file, index=False)

                print(
                    f"Aggregated {model_name} for {shot}-shot "
                    f"using {len(clean_sets)} set(s)"
                )

## 8. Experiment 2 - Small Fine-Tuned model vs. larger ICL model

This experiment compares full-data Fine-Tuned Qwen 0.5B with the larger Gemma 2B model under all ICL shot settings.

In [ ]:
if RUN_EXPERIMENT_2:
    small_ft_adapter = "Qwen_Qwen2.5-0.5B-Instruct_100_percent"
    small_ft_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    larger_icl_model = "google/gemma-2-2b-it"
    labels = ["left", "lean left", "center", "lean right", "right"]

    if "ft_predictions" not in globals():
        raise RuntimeError(
            "Experiment 2 requires ft_predictions. Run the Fine-tuned evaluation first."
        )

    if "icl_predictions" not in globals():
        raise RuntimeError(
            "Experiment 2 requires icl_predictions. Run ICL and ICL aggregation first."
        )

    if small_ft_adapter not in ft_predictions:
        raise KeyError(f"Missing Fine-tuned predictions for {small_ft_adapter}.")

    if larger_icl_model not in icl_predictions:
        raise KeyError(f"Missing ICL predictions for {larger_icl_model}.")

    experiment_2_rows = []
    small_ft_preds = ft_predictions[small_ft_adapter]

    if len(small_ft_preds) != len(y_test):
        raise ValueError("The Fine-tuned prediction count does not match y_test.")

    experiment_2_rows.append({
        "Method": "Fine-Tuning",
        "Model": small_ft_model_name,
        "Training Data": "100%",
        "Shots": "N/A",
        "Accuracy": accuracy_score(y_test, small_ft_preds),
        "Macro-F1": f1_score(y_test, small_ft_preds, average="macro", labels=labels),
        "Inference Time Seconds": (
            ft_inference_times.get(small_ft_adapter)
            if "ft_inference_times" in globals()
            else None
        ),
    })

    for shot in shots:
        if shot not in icl_predictions[larger_icl_model]:
            continue

        larger_icl_preds = icl_predictions[larger_icl_model][shot]

        if len(larger_icl_preds) != len(y_test):
            raise ValueError(f"The Gemma {shot}-shot prediction count does not match y_test.")

        gemma_inference_time = None

        if "icl_inference_times" in globals():
            time_prefix = f"{larger_icl_model}_{shot}_shot_"
            matching_times = [
                seconds
                for time_key, seconds in icl_inference_times.items()
                if time_key.startswith(time_prefix)
            ]

            if matching_times:
                gemma_inference_time = sum(matching_times)

        experiment_2_rows.append({
            "Method": "ICL",
            "Model": larger_icl_model,
            "Training Data": "None",
            "Shots": shot,
            "Accuracy": accuracy_score(y_test, larger_icl_preds),
            "Macro-F1": f1_score(y_test, larger_icl_preds, average="macro", labels=labels),
            "Inference Time Seconds": gemma_inference_time,
        })

    experiment_2_df = pd.DataFrame(experiment_2_rows)

    print("\nExperiment 2: Small Fine-Tuned Model vs Larger ICL Model")
    print(experiment_2_df.to_string(index=False))

    experiment_2_df.to_csv(RESULTS_DIR / "experiment_2_results.csv", index=False)
    print("\nExperiment 2 results saved to results/experiment_2_results.csv")

## 9. Evaluation metrics and confusion matrices

Accuracy and Macro-F1 are computed on the 690-example test set. Macro-F1 is the main metric because the five classes are imbalanced.

In [ ]:
if RUN_EVALUATION:
    label_order = ["left", "lean left", "center", "lean right", "right"]

    def validate_prediction_length(experiment_name, predictions):
        if len(predictions) != len(y_test):
            raise ValueError(
                f"{experiment_name}: expected {len(y_test)} predictions, "
                f"but received {len(predictions)}."
            )

    def calculate_invalid_rate(predictions):
        if not predictions:
            return 0.0

        return sum(prediction == "invalid" for prediction in predictions) / len(predictions)

    def get_ft_inference_time(model_name):
        if "ft_inference_times" not in globals():
            return None

        return ft_inference_times.get(model_name)

    def get_icl_inference_time(model_name, shot):
        if "icl_inference_times" not in globals():
            return None

        prefix = f"{model_name}_{shot}_shot_"
        matching_times = [
            seconds
            for time_key, seconds in icl_inference_times.items()
            if time_key.startswith(prefix)
        ]

        return sum(matching_times) if matching_times else None

    all_metrics = []
    confusion_matrices = {}

    validate_prediction_length("Majority Class Baseline", dummy_preds)

    all_metrics.append({
        "Method": "Baseline",
        "Model": "Majority Class",
        "Training Data": "100%",
        "Shots": "N/A",
        "Accuracy": accuracy_score(y_test, dummy_preds),
        "Macro-F1": f1_score(
            y_test, dummy_preds, average="macro", labels=label_order
        ),
        "Invalid Rate": 0.0,
        "Inference Time Seconds": None,
        "Seconds Per Article": None,
    })

    confusion_matrices["Majority Class Baseline"] = confusion_matrix(
        y_test, dummy_preds, labels=label_order
    )

    validate_prediction_length("TF-IDF + Logistic Regression", lr_preds)

    all_metrics.append({
        "Method": "Baseline",
        "Model": "TF-IDF + Logistic Regression",
        "Training Data": "100%",
        "Shots": "N/A",
        "Accuracy": accuracy_score(y_test, lr_preds),
        "Macro-F1": f1_score(
            y_test, lr_preds, average="macro", labels=label_order
        ),
        "Invalid Rate": 0.0,
        "Inference Time Seconds": None,
        "Seconds Per Article": None,
    })

    confusion_matrices["TF-IDF + Logistic Regression"] = confusion_matrix(
        y_test, lr_preds, labels=label_order
    )

    if "ft_predictions" in globals():
        for model_name, preds in ft_predictions.items():
            validate_prediction_length(model_name, preds)

            training_data = "Unknown"

            for train_size in train_sizes:
                size_text = f"_{int(train_size * 100)}_percent"

                if model_name.endswith(size_text):
                    training_data = f"{int(train_size * 100)}%"
                    break

            inference_time = get_ft_inference_time(model_name)
            seconds_per_article = (
                inference_time / len(y_test) if inference_time is not None else None
            )

            all_metrics.append({
                "Method": "Fine-Tuning",
                "Model": model_name,
                "Training Data": training_data,
                "Shots": "N/A",
                "Accuracy": accuracy_score(y_test, preds),
                "Macro-F1": f1_score(
                    y_test, preds, average="macro", labels=label_order
                ),
                "Invalid Rate": calculate_invalid_rate(preds),
                "Inference Time Seconds": inference_time,
                "Seconds Per Article": seconds_per_article,
            })

            confusion_matrices[model_name] = confusion_matrix(
                y_test, preds, labels=label_order
            )

    if "icl_predictions" in globals():
        for base_model, shots_dict in icl_predictions.items():
            for shot, preds in shots_dict.items():
                exp_name = f"{base_model}_{shot}-shots"
                validate_prediction_length(exp_name, preds)

                inference_time = get_icl_inference_time(base_model, shot)
                seconds_per_article = (
                    inference_time / len(y_test) if inference_time is not None else None
                )

                all_metrics.append({
                    "Method": "ICL",
                    "Model": base_model,
                    "Training Data": "None",
                    "Shots": shot,
                    "Accuracy": accuracy_score(y_test, preds),
                    "Macro-F1": f1_score(
                        y_test, preds, average="macro", labels=label_order
                    ),
                    "Invalid Rate": calculate_invalid_rate(preds),
                    "Inference Time Seconds": inference_time,
                    "Seconds Per Article": seconds_per_article,
                })

                confusion_matrices[exp_name] = confusion_matrix(
                    y_test, preds, labels=label_order
                )

    metrics_df = pd.DataFrame(all_metrics)
    metrics_df["Shots Sort"] = pd.to_numeric(
        metrics_df["Shots"], errors="coerce"
    ).fillna(-1)

    metrics_df = metrics_df.sort_values(
        by=["Method", "Model", "Shots Sort"],
        ignore_index=True,
    ).drop(columns=["Shots Sort"])

    print("\nEvaluation metrics:")
    print(metrics_df.to_string(index=False))

    metrics_df.to_csv(RESULTS_DIR / "all_metrics.csv", index=False)
    print("\nMetrics saved to results/all_metrics.csv")

## 10. Visualizations

The following functions generate the final summary plots and confusion matrices. The direct Qwen Fine-Tuning vs. Qwen ICL plot is included because it is the central comparison in the final report.

In [ ]:
sns.set_theme(style="whitegrid")


def save_and_show(filename):
    """Save a plot to the repository and display it in the notebook."""
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def get_short_model_name(model_name):
    """Convert internal model identifiers to short plot labels."""
    model_name = str(model_name)

    if "Qwen2.5-0.5B" in model_name:
        return "Qwen 0.5B"

    if "gemma-2-2b" in model_name.lower():
        return "Gemma 2B"

    model_name = re.sub(r"_\d+_percent$", "", model_name)
    return model_name.replace("_", " ")

In [ ]:
def plot_fine_tuning(metrics_df, lr_f1, dummy_f1):
    ft_plot_df = metrics_df[metrics_df["Method"] == "Fine-Tuning"].copy()
    if ft_plot_df.empty:
        return

    ft_plot_df["Training Percentage"] = pd.to_numeric(
        ft_plot_df["Training Data"].astype(str).str.replace("%", "", regex=False),
        errors="coerce",
    )
    ft_plot_df = ft_plot_df.dropna(subset=["Training Percentage", "Macro-F1"])
    ft_plot_df["Base Model"] = ft_plot_df["Model"].map(get_short_model_name)
    ft_plot_df = ft_plot_df.sort_values(["Base Model", "Training Percentage"])

    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=ft_plot_df,
        x="Training Percentage",
        y="Macro-F1",
        hue="Base Model",
        marker="o",
        errorbar=None,
    )
    plt.axhline(
        y=lr_f1,
        linestyle="--",
        linewidth=2,
        label=f"TF-IDF + Logistic Regression ({lr_f1:.3f})",
    )
    plt.axhline(
        y=dummy_f1,
        linestyle=":",
        linewidth=2,
        label=f"Majority Class Baseline ({dummy_f1:.3f})",
    )
    plt.title("Fine-Tuning Performance by Training Data Size")
    plt.xlabel("Training Data (%)")
    plt.ylabel("Macro-F1")
    plt.xticks(sorted(ft_plot_df["Training Percentage"].unique()))
    plt.ylim(0, 1)
    plt.grid(axis="y", alpha=0.3)
    plt.legend(title="Model / Baseline")

    save_and_show("fine_tuning_by_data_size.png")

In [ ]:
def plot_icl(metrics_df):
    icl_plot_df = metrics_df[metrics_df["Method"] == "ICL"].copy()
    if icl_plot_df.empty:
        return

    icl_plot_df["Shots"] = pd.to_numeric(icl_plot_df["Shots"], errors="coerce")
    icl_plot_df = icl_plot_df.dropna(subset=["Shots"])
    icl_plot_df["Model Label"] = icl_plot_df["Model"].map(get_short_model_name)

    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=icl_plot_df,
        x="Shots",
        y="Macro-F1",
        hue="Model Label",
        marker="o",
        errorbar=None,
    )
    plt.title("ICL Performance by Number of Demonstrations")
    plt.xlabel("Number of ICL Demonstrations")
    plt.ylabel("Macro-F1")
    plt.xticks(sorted(icl_plot_df["Shots"].unique()))

    save_and_show("icl_by_shots.png")

In [ ]:
def plot_qwen_ft_vs_icl(metrics_df):
    """Directly compare the two adaptation methods on the same Qwen 0.5B model."""
    qwen_ft = metrics_df[
        (metrics_df["Method"] == "Fine-Tuning")
        & metrics_df["Model"].astype(str).str.contains("Qwen", na=False)
    ].copy()

    qwen_icl = metrics_df[
        (metrics_df["Method"] == "ICL")
        & metrics_df["Model"].astype(str).str.contains("Qwen", na=False)
    ].copy()

    if qwen_ft.empty or qwen_icl.empty:
        return

    qwen_ft["Sort Value"] = pd.to_numeric(
        qwen_ft["Training Data"].astype(str).str.replace("%", "", regex=False),
        errors="coerce",
    )
    qwen_ft = qwen_ft.sort_values("Sort Value")

    qwen_icl["Sort Value"] = pd.to_numeric(qwen_icl["Shots"], errors="coerce")
    qwen_icl = qwen_icl.sort_values("Sort Value")

    labels = (
        [f"FT {value}" for value in qwen_ft["Training Data"]]
        + [f"ICL {int(value)}-shot" for value in qwen_icl["Shots"]]
    )
    values = qwen_ft["Macro-F1"].tolist() + qwen_icl["Macro-F1"].tolist()
    positions = list(range(len(qwen_ft))) + list(
        range(len(qwen_ft) + 1, len(qwen_ft) + 1 + len(qwen_icl))
    )

    plt.figure(figsize=(11, 6))
    bars = plt.bar(positions, values)
    plt.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    plt.axvline(len(qwen_ft) - 0.5, linestyle="--", linewidth=1)
    plt.xticks(positions, labels, rotation=35, ha="right")
    plt.ylabel("Macro-F1")
    plt.title("Qwen 0.5B: Fine-Tuning vs. In-Context Learning")
    plt.ylim(0, max(values) * 1.18)

    save_and_show("qwen_ft_vs_icl_direct_comparison.png")

In [ ]:
def plot_efficiency(metrics_df):
    efficiency_df = metrics_df.dropna(
        subset=["Inference Time Seconds", "Macro-F1"]
    ).copy()

    if efficiency_df.empty:
        return

    efficiency_df["Method"] = efficiency_df["Method"].replace(
        {"ICL": "In-Context Learning"}
    )

    plt.figure(figsize=(11, 7))
    sns.scatterplot(
        data=efficiency_df,
        x="Inference Time Seconds",
        y="Macro-F1",
        hue="Method",
        style="Method",
        s=140,
    )

    efficiency_df["Configuration"] = efficiency_df.apply(
        lambda row: (
            str(row["Training Data"])
            if row["Method"] == "Fine-Tuning"
            else f"{int(row['Shots'])}-shot"
        ),
        axis=1,
    )

    for _, row in efficiency_df.iterrows():
        plt.annotate(
            row["Configuration"],
            (row["Inference Time Seconds"], row["Macro-F1"]),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=9,
        )

    plt.title("Predictive Quality vs. Inference Time")
    plt.xlabel("Inference Time on Test Set (seconds)")
    plt.ylabel("Macro-F1")
    plt.grid(alpha=0.3)

    save_and_show("quality_vs_inference_time.png")

In [ ]:
def plot_experiment_2(experiment_2_df):
    if experiment_2_df.empty:
        return

    plot_df = experiment_2_df.copy()
    plot_df["Configuration"] = plot_df.apply(
        lambda row: (
            "Qwen 0.5B Fine-Tuned"
            if row["Method"] == "Fine-Tuning"
            else f"Gemma 2B ICL ({row['Shots']}-shot)"
        ),
        axis=1,
    )

    plt.figure(figsize=(11, 6))
    ax = sns.barplot(
        data=plot_df,
        x="Configuration",
        y="Macro-F1",
        errorbar=None,
    )

    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=3)

    plt.title("Small Fine-Tuned Model vs. Larger ICL Model")
    plt.xlabel("")
    plt.ylabel("Macro-F1")
    plt.xticks(rotation=35, ha="right")

    save_and_show("experiment_2_comparison.png")

In [ ]:
def plot_confusion_matrices(confusion_matrices, label_order):
    for experiment_name, matrix in confusion_matrices.items():
        plt.figure(figsize=(6, 5))
        sns.heatmap(
            matrix,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=label_order,
            yticklabels=label_order,
        )
        plt.title(f"Confusion Matrix: {experiment_name}")
        plt.ylabel("True Label")
        plt.xlabel("Predicted Label")

        safe_name = str(experiment_name).replace("/", "_").replace(" ", "_")
        save_and_show(f"cm_{safe_name}.png")

In [ ]:
if RUN_EVALUATION and "metrics_df" in globals() and not metrics_df.empty:
    plot_fine_tuning(metrics_df, lr_f1, dummy_f1)
    plot_icl(metrics_df)
    plot_qwen_ft_vs_icl(metrics_df)
    plot_efficiency(metrics_df)

    if "experiment_2_df" in globals():
        plot_experiment_2(experiment_2_df)

    if "confusion_matrices" in globals():
        plot_confusion_matrices(confusion_matrices, LABEL_ORDER)

    print(f"Visualizations saved to: {PLOTS_DIR}")

## 11. Error analysis

This final section prints the most frequent confusion pairs and one example error for every evaluated model configuration. It is intended for qualitative inspection and does not change any reported metric.

In [ ]:
if RUN_EVALUATION:
    def analyze_errors(model_name, true_labels, predicted_labels, texts):
        df = pd.DataFrame({
            'Text': texts,
            'True': true_labels,
            'Predicted': predicted_labels
        })
        errors = df[df['True'] != df['Predicted']]

        if errors.empty:
            return

        print(f"\n=== Error Analysis: {model_name} ===")
        confusions = errors.groupby(['True', 'Predicted']).size().reset_index(name='Count')
        confusions = confusions.sort_values(by='Count', ascending=False)
        print("Top 3 Confusions:")
        print(confusions.head(3).to_string(index=False))

        print("\nExample Error:")
        example = errors.iloc[0]
        print(f"True: {example['True']} | Predicted: {example['Predicted']}")
        print(f"Text snippet: {str(example['Text']).strip()[:200]}...\n")

    if 'ft_predictions' in globals():
        for m_name, preds in ft_predictions.items():
            analyze_errors(f"FT - {m_name}", y_test.values, preds, X_test.values)

    if 'icl_predictions' in globals():
        for m_name, shots_dict in icl_predictions.items():
            for shot, preds in shots_dict.items():
                analyze_errors(f"ICL - {m_name} ({shot} shots)", y_test.values, preds, X_test.values)

## Reproducibility notes

- The final project used seed 42 and a stratified 70/15/15 split.
- Fine-Tuning used Qwen2.5-0.5B-Instruct with LoRA on 1%, 5%, 10%, 25%, 50%, and 100% of the training split.
- ICL used Qwen2.5-0.5B-Instruct and Gemma 2 2B with 0, 1, 3, 5, and 10 demonstrations.
- The prediction parser and ICL aggregation rule are intentionally preserved exactly as used for the reported experiment results.
- Runtime values in the repository were collected across NVIDIA L4 and T4 sessions and should not be treated as a controlled hardware comparison.